# RQ2: Financial Predictors of High Default Risk

This Kaggle notebook produces an actual odds-ratio table and forest plot for financial predictors of high default risk.

**Outputs saved to `/kaggle/working/rq2_financial_predictors/`:**
- `RQ2_financial_predictors_odds_ratio_table.csv`
- `RQ2_financial_predictors_forest_plot.pdf`

**Research question:** Which financial indicators are the strongest predictors of high default risk among EV charging users?

## Run this cell to generate the actual answer, CSV table, and PDF figure

The notebook automatically searches for the dataset file inside `/kaggle/input`. If it cannot find it, set `DATASET_PATH` manually in the code cell.

In [ ]:

# ============================================================
# Common setup: imports, paths, loading, cleaning, preprocessing
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)

RANDOM_STATE = 42

# Change this manually only if Kaggle cannot auto-detect your file.
# Example:
# DATASET_PATH = "/kaggle/input/ev-charging-behavior-analysis-and-financial-risk/your_file.csv"
DATASET_PATH = None

def file_has_expected_columns(path):
    """
    Checks whether a candidate file looks like the raw EV dataset rather than
    an output table generated by one of the notebooks.
    """
    try:
        if path.lower().endswith(".csv"):
            cols = pd.read_csv(path, nrows=0).columns.astype(str).str.strip().tolist()
        elif path.lower().endswith((".xlsx", ".xls")):
            cols = pd.read_excel(path, nrows=0).columns.astype(str).str.strip().tolist()
        else:
            return False
    except Exception:
        return False

    required = {"High_Default_Risk", "Charging_Efficiency_Index"}
    return required.issubset(set(cols)) and len(cols) >= 10

def find_dataset_file():
    """
    Finds the raw CSV or Excel dataset file automatically.
    Works in Kaggle by searching /kaggle/input first.
    Also avoids accidentally selecting CSV output tables generated by the notebooks.
    """
    if DATASET_PATH is not None and os.path.exists(DATASET_PATH):
        return DATASET_PATH

    roots = ["/kaggle/input", ".", "/mnt/data"]
    extensions = (".csv", ".xlsx", ".xls")
    candidates = []

    for root in roots:
        if not os.path.exists(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            # Skip common output folders when running locally
            if any(part.startswith("rq") and "output" in part for part in dirpath.lower().split(os.sep)):
                continue
            for filename in filenames:
                lower = filename.lower()
                if lower.endswith(extensions):
                    candidates.append(os.path.join(dirpath, filename))

    if not candidates:
        raise FileNotFoundError(
            "No CSV/XLS/XLSX file found. Add the Kaggle dataset to the notebook "
            "or set DATASET_PATH manually."
        )

    # First priority: files that actually contain the required raw dataset columns.
    valid_raw_files = [path for path in candidates if file_has_expected_columns(path)]
    if valid_raw_files:
        keywords = ["ev", "charging", "financial", "risk", "behavior"]
        preferred = [
            path for path in valid_raw_files
            if any(word in os.path.basename(path).lower() for word in keywords)
        ]
        return sorted(preferred or valid_raw_files)[0]

    # Fallback: use filename keywords if header check cannot be completed.
    keywords = ["ev", "charging", "financial", "risk", "behavior"]
    preferred = [
        path for path in candidates
        if any(word in os.path.basename(path).lower() for word in keywords)
    ]

    selected = sorted(preferred or candidates)[0]
    return selected

def load_raw_dataset():
    path = find_dataset_file()
    print(f"Loading dataset from: {path}")

    if path.lower().endswith(".csv"):
        df = pd.read_csv(path)
    elif path.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(path)
    else:
        raise ValueError("Unsupported file type. Use CSV, XLSX, or XLS.")

    print(f"Raw dataset shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    return df

def clean_dataset(df):
    """
    Basic cleaning for this EV charging dataset:
    - strips column names and string values
    - removes duplicate rows
    - converts known numeric columns
    - converts impossible negative values into missing values
    - standardizes binary columns where needed
    """
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    df = df.drop_duplicates()

    # Strip text columns
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"nan": np.nan, "None": np.nan, "": np.nan})

    # Standardize binary columns if they came in as text before numeric conversion
    binary_maps = {
        "yes": 1, "y": 1, "true": 1, "high": 1, "risk": 1, "default": 1, "1": 1,
        "no": 0, "n": 0, "false": 0, "low": 0, "non-risk": 0, "non risk": 0, "0": 0
    }
    for col in ["Loan_Taken", "High_Default_Risk"]:
        if col in df.columns and df[col].dtype == "object":
            df[col] = df[col].astype(str).str.lower().map(binary_maps)

    known_numeric = [
        "User_ID", "Age", "Battery_Capacity_kWh", "Charging_Sessions_Per_Month",
        "Avg_Charge_Cost", "Distance_Travelled_Per_Month", "Loan_Taken",
        "Missed_Payments_Last_6M", "Tenure_Months", "App_Usage_Score",
        "Charging_Time_Minutes", "High_Default_Risk", "Charging_Efficiency_Index"
    ]
    for col in known_numeric:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Treat impossible negative values as missing
    non_negative_cols = [
        "Age", "Battery_Capacity_kWh", "Charging_Sessions_Per_Month",
        "Avg_Charge_Cost", "Distance_Travelled_Per_Month",
        "Missed_Payments_Last_6M", "Tenure_Months", "App_Usage_Score",
        "Charging_Time_Minutes", "Charging_Efficiency_Index"
    ]
    for col in non_negative_cols:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan

    # Keep app score in a practical range if the column exists
    if "App_Usage_Score" in df.columns:
        df.loc[(df["App_Usage_Score"] < 0) | (df["App_Usage_Score"] > 10), "App_Usage_Score"] = np.nan

    return df

def make_output_dir(name):
    output_dir = os.path.join("/kaggle/working" if os.path.exists("/kaggle/working") else ".", name)
    os.makedirs(output_dir, exist_ok=True)
    print(f"Output folder: {output_dir}")
    return output_dir

def split_features_target(df, target_col, drop_cols=None):
    if drop_cols is None:
        drop_cols = []
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found. Available columns: {list(df.columns)}")

    data = df.dropna(subset=[target_col]).copy()

    # Force target to integer binary if classification
    if target_col == "High_Default_Risk":
        data = data[data[target_col].isin([0, 1])].copy()
        data[target_col] = data[target_col].astype(int)

    y = data[target_col]
    X = data.drop(columns=[target_col] + [c for c in drop_cols if c in data.columns])
    return X, y, data

def get_feature_types(X):
    categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numeric_cols = [c for c in X.columns if c not in categorical_cols]
    return numeric_cols, categorical_cols

def make_preprocessor(X, scale_numeric=True):
    numeric_cols, categorical_cols = get_feature_types(X)

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_transformer = Pipeline(steps=numeric_steps)

    try:
        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ])
    except TypeError:
        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))
        ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols)
        ],
        remainder="drop"
    )
    return preprocessor, numeric_cols, categorical_cols

def get_transformed_feature_names(preprocessor, numeric_cols, categorical_cols):
    feature_names = list(numeric_cols)
    if categorical_cols:
        try:
            ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
            cat_names = list(ohe.get_feature_names_out(categorical_cols))
        except Exception:
            cat_names = []
            for col in categorical_cols:
                cat_names.append(col)
        feature_names.extend(cat_names)
    return feature_names

def save_table(df, output_dir, filename):
    path = os.path.join(output_dir, filename)
    df.to_csv(path, index=False)
    print(f"Saved table: {path}")
    return path

def save_figure(output_dir, filename):
    path = os.path.join(output_dir, filename)
    plt.savefig(path, format="pdf", bbox_inches="tight")
    print(f"Saved figure: {path}")
    return path

def style_plot(title, subtitle=None, xlabel=None, ylabel=None):
    plt.title(title, fontsize=15, fontweight="bold", pad=18)
    if subtitle:
        plt.suptitle(subtitle, fontsize=10, y=0.94, color="#3b4f6b")
    if xlabel:
        plt.xlabel(xlabel, fontsize=11)
    if ylabel:
        plt.ylabel(ylabel, fontsize=11)
    plt.grid(axis="y", linestyle="--", alpha=0.35)
    plt.gca().set_axisbelow(True)

def print_dataset_summary(df):
    print("Cleaned dataset shape:", df.shape)
    print("\nColumns:")
    print(list(df.columns))
    print("\nMissing values:")
    display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count").head(20))


# ============================================================
# RQ2: Financial predictors of high default risk
# ============================================================

output_dir = make_output_dir("rq2_financial_predictors")

df_raw = load_raw_dataset()
df = clean_dataset(df_raw)
print_dataset_summary(df)

target_col = "High_Default_Risk"

candidate_predictors = [
    "Missed_Payments_Last_6M",
    "Loan_Taken",
    "Income_Level",
    "Tenure_Months",
    "Avg_Charge_Cost",
    "App_Usage_Score",
    "Charging_Efficiency_Index"
]

predictors = [c for c in candidate_predictors if c in df.columns]
if len(predictors) < 2:
    raise ValueError("Not enough expected financial/product predictors were found in the dataset.")

data = df.dropna(subset=[target_col]).copy()
data = data[data[target_col].isin([0, 1])].copy()
data[target_col] = data[target_col].astype(int)

model_df = data[predictors + [target_col]].copy()

# Prepare a statsmodels logistic regression design matrix
# drop_first=True gives interpretable reference categories.
X_design = pd.get_dummies(model_df[predictors], drop_first=True, dtype=float)
y = model_df[target_col].astype(int)

# Impute missing values after dummy encoding
for col in X_design.columns:
    X_design[col] = pd.to_numeric(X_design[col], errors="coerce")
    X_design[col] = X_design[col].fillna(X_design[col].median())

try:
    import statsmodels.api as sm

    X_sm = sm.add_constant(X_design, has_constant="add")
    logit_model = sm.Logit(y, X_sm)
    fit = logit_model.fit(disp=False, maxiter=300)

    params = fit.params.drop("const", errors="ignore")
    conf = fit.conf_int().drop("const", errors="ignore")
    pvalues = fit.pvalues.drop("const", errors="ignore")

    results = pd.DataFrame({
        "Predictor": params.index,
        "Log_Odds": params.values,
        "Adjusted_Odds_Ratio": np.exp(params.values),
        "CI_Lower": np.exp(conf[0].values),
        "CI_Upper": np.exp(conf[1].values),
        "P_Value": pvalues.values
    })

except Exception as e:
    print("Statsmodels logistic regression failed, using bootstrap-based sklearn fallback.")
    print("Reason:", e)

    from sklearn.linear_model import LogisticRegression
    from sklearn.utils import resample

    base_model = LogisticRegression(max_iter=3000, class_weight="balanced")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_design)
    base_model.fit(X_scaled, y)
    coef = base_model.coef_[0]

    boot_coefs = []
    rng = np.random.default_rng(RANDOM_STATE)
    for _ in range(100):
        idx = rng.choice(np.arange(len(y)), size=len(y), replace=True)
        try:
            m = LogisticRegression(max_iter=3000, class_weight="balanced")
            m.fit(X_scaled[idx], y.iloc[idx])
            boot_coefs.append(m.coef_[0])
        except Exception:
            continue

    boot_coefs = np.array(boot_coefs)
    ci_lower = np.percentile(boot_coefs, 2.5, axis=0)
    ci_upper = np.percentile(boot_coefs, 97.5, axis=0)

    results = pd.DataFrame({
        "Predictor": X_design.columns,
        "Log_Odds": coef,
        "Adjusted_Odds_Ratio": np.exp(coef),
        "CI_Lower": np.exp(ci_lower),
        "CI_Upper": np.exp(ci_upper),
        "P_Value": np.nan
    })

results["Direction"] = np.where(results["Adjusted_Odds_Ratio"] > 1, "Increases risk", "Reduces risk")
results = results.sort_values("Adjusted_Odds_Ratio", ascending=False).reset_index(drop=True)

results_rounded = results.copy()
for col in ["Log_Odds", "Adjusted_Odds_Ratio", "CI_Lower", "CI_Upper", "P_Value"]:
    results_rounded[col] = results_rounded[col].round(4)

display(results_rounded)
save_table(results_rounded, output_dir, "RQ2_financial_predictors_odds_ratio_table.csv")

top_risk = results_rounded.iloc[0]
print(
    f"Actual answer for RQ2: The strongest risk-increasing predictor is {top_risk['Predictor']} "
    f"with adjusted odds ratio = {top_risk['Adjusted_Odds_Ratio']}."
)

# Figure 2: Forest plot of odds ratios
plot_df = results.copy().sort_values("Adjusted_Odds_Ratio", ascending=True)

plt.figure(figsize=(10, max(6, 0.45 * len(plot_df))))
y_pos = np.arange(len(plot_df))

or_values = plot_df["Adjusted_Odds_Ratio"].values
lower_error = or_values - plot_df["CI_Lower"].values
upper_error = plot_df["CI_Upper"].values - or_values

plt.errorbar(
    or_values,
    y_pos,
    xerr=[lower_error, upper_error],
    fmt="o",
    color="#1f5fbf",
    ecolor="#1f5fbf",
    elinewidth=1.5,
    capsize=4
)

plt.axvline(1.0, color="#333333", linestyle="--", linewidth=1)
plt.yticks(y_pos, plot_df["Predictor"])
plt.xlabel("Adjusted odds ratio")
plt.title("Figure 2. Financial predictors of high default risk", fontsize=15, fontweight="bold", pad=18)
plt.grid(axis="x", linestyle="--", alpha=0.35)

for y_i, row in enumerate(plot_df.itertuples()):
    plt.text(
        row.CI_Upper + 0.03,
        y_i,
        f"{row.Adjusted_Odds_Ratio:.2f} ({row.CI_Lower:.2f}–{row.CI_Upper:.2f})",
        va="center",
        fontsize=9
    )

plt.figtext(
    0.5, -0.02,
    "Note: Odds ratios above 1 indicate higher default risk; values below 1 indicate lower risk.",
    ha="center", fontsize=9
)
plt.tight_layout()

save_figure(output_dir, "RQ2_financial_predictors_forest_plot.pdf")
plt.show()
